# 🚦 ENHANCED Traffic Violation Model Training

**Trains YOLOv8 model for ALL violations:**
- 🪖 Helmet / No-Helmet detection
- 🚦 Traffic Lights (Red/Yellow/Green)
- 🚗 All vehicle types
- 🔢 Number plates

**Datasets Used:**
1. Smart Helmet Detection (Kaggle)
2. LISA Traffic Light Dataset (Kaggle)

---

## Step 1: Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

# Install dependencies
!pip install -q ultralytics kaggle

from IPython import display
display.clear_output()

import torch
print("✅ Environment ready!")
print("🎮 GPU:", "Available" if torch.cuda.is_available() else "Not available (CPU mode)")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"⚡ CUDA: {torch.cuda.is_available() Disclaimer: Training will be MUCH slower on CPU!")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

output_dir = '/content/drive/MyDrive/traffic-violation-model-enhanced'
os.makedirs(output_dir, exist_ok=True)

print(f"✅ Google Drive mounted!")
print(f"📁 Model saves to: {output_dir}")

## Step 3: Setup Kaggle & Download Datasets

In [ ]:
# Upload kaggle.json
from google.colab import files

print("📤 Upload your kaggle.json (from Kaggle → Settings → API)")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✅ Kaggle credentials configured!")

In [ ]:
# Download Dataset 1: Helmet Detection
print("📥 Downloading Helmet Detection Dataset...")
!kaggle datasets download -d aditya4567876543456/smart-helmet-detection-using-yolo-v8
!unzip -q smart-helmet-detection-using-yolo-v8.zip -d /content/helmet-dataset
print("✅ Helmet dataset downloaded!")

In [ ]:
# Download Dataset 2: Traffic Lights
print("📥 Downloading Traffic Light Dataset...")
!kaggle datasets download -d zulaikhaamirova6/lisa-traffic-light-datasetyolo
!unzip -q lisa-traffic-light-datasetyolo.zip -d /content/traffic-light-dataset
print("✅ Traffic light dataset downloaded!")

## Step 4: Merge Datasets

In [ ]:
import os
import shutil
import glob

# Create merged dataset structure
merged_dir = '/content/merged-dataset'
os.makedirs(f'{merged_dir}/train/images', exist_ok=True)
os.makedirs(f'{merged_dir}/train/labels', exist_ok=True)
os.makedirs(f'{merged_dir}/valid/images', exist_ok=True)
os.makedirs(f'{merged_dir}/valid/labels', exist_ok=True)

# Copy helmet dataset files
print("📋 Merging helmet dataset...")
helmet_train_images = glob.glob('/content/helmet-dataset/**/train/images/*', recursive=True)
helmet_train_labels = glob.glob('/content/helmet-dataset/**/train/labels/*', recursive=True)

for img in helmet_train_images:
    shutil.copy(img, f'{merged_dir}/train/images/')
for lbl in helmet_train_labels:
    shutil.copy(lbl, f'{merged_dir}/train/labels/')

helmet_valid_images = glob.glob('/content/helmet-dataset/**/valid/images/*', recursive=True)
helmet_valid_labels = glob.glob('/content/helmet-dataset/**/valid/labels/*', recursive=True)

for img in helmet_valid_images:
    shutil.copy(img, f'{merged_dir}/valid/images/')
for lbl in helmet_valid_labels:
    shutil.copy(lbl, f'{merged_dir}/valid/labels/')

print(f"✅ Helmet: {len(helmet_train_images)} train, {len(helmet_valid_images)} valid")

# Copy traffic light dataset
print("📋 Merging traffic light dataset...")
traffic_train_images = glob.glob('/content/traffic-light-dataset/**/train/images/*', recursive=True)
traffic_train_labels = glob.glob('/content/traffic-light-dataset/**/train/labels/*', recursive=True)

for img in traffic_train_images:
    shutil.copy(img, f'{merged_dir}/train/images/')
for lbl in traffic_train_labels:
    shutil.copy(lbl, f'{merged_dir}/train/labels/')

traffic_valid_images = glob.glob('/content/traffic-light-dataset/**/valid/images/*', recursive=True)
traffic_valid_labels = glob.glob('/content/traffic-light-dataset/**/valid/labels/*', recursive=True)

for img in traffic_valid_images:
    shutil.copy(img, f'{merged_dir}/valid/images/')
for lbl in traffic_valid_labels:
    shutil.copy(lbl, f'{merged_dir}/valid/labels/')

print(f"✅ Traffic: {len(traffic_train_images)} train, {len(traffic_valid_images)} valid")

total_train = len(glob.glob(f'{merged_dir}/train/images/*'))
total_valid = len(glob.glob(f'{merged_dir}/valid/images/*'))

print(f"\n📊 Total Merged Dataset:")
print(f"  Training images: {total_train}")
print(f"  Validation images: {total_valid}")

In [ ]:
# Create combined data.yaml
data_yaml_content = f'''path: {merged_dir}
train: train/images
val: valid/images

nc: 5
names:
  0: with_helmet
  1: without_helmet
  2: traffic-light-red
  3: traffic-light-yellow
  4: traffic-light-green
'''

data_yaml_path = f'{merged_dir}/data.yaml'
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print("✅ Created data.yaml")
print("\n📄 Configuration:")
print(data_yaml_content)

## Step 5: Train Enhanced Model 🚀

In [ ]:
from ultralytics import YOLO
import time

model = YOLO('yolov8s.pt')

print("🚀 Starting training...")
print("⏱️ Estimated time: ~1-2 hours")
print("\n🎯 Model will detect:")
print("  ✅ Helmet violations")
print("  ✅ Red light jumping  ")
print("  ✅ Triple riding")
print("  ✅ All vehicle types\n")

start_time = time.time()

results = model.train(
    data=data_yaml_path,
    epochs=100,             # 100 epochs for better accuracy
    imgsz=640,
    batch=16,
    name='traffic-violation-complete',
    patience=20,
    save=True,
    device=0,
    workers=8,
    optimizer='AdamW',
    lr0=0.01,
    weight_decay=0.0005,
    augment=True,
    plots=True,
    project='/content/runs',
    exist_ok=True
)

training_time = time.time() - start_time

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)
print(f"⏱️ Training time: {training_time/60:.1f} minutes")

## Step 6: Validate Model

In [ ]:
metrics = model.val()

print("\n📊 Model Performance:")
print("="*50)
print(f"  mAP50:      {metrics.box.map50:.4f}")
print(f"  mAP50-95:   {metrics.box.map:.4f}")
print(f"  Precision:  {metrics.box.mp:.4f}")
print(f"  Recall:     {metrics.box.mr:.4f}")
print("="*50)

## Step 7: Save to Google Drive ⬇️

In [ ]:
import shutil

best_model_path = '/content/runs/detect/traffic-violation-complete/weights/best.pt'
shutil.copy(best_model_path, f'{output_dir}/best.pt')

# Save training results
results_dir = '/content/runs/detect/traffic-violation-complete'
for file in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    if os.path.exists(f'{results_dir}/{file}'):
        shutil.copy(f'{results_dir}/{file}', f'{output_dir}/{file}')

print("\n" + "="*60)
print("✅ ENHANCED MODEL SAVED!")
print("="*60)
print(f"\n📁 Location: {output_dir}/")
print("\n📥 Download: best.pt")
print("\n🚀 DEPLOYMENT:")
print("  1. Download best.pt from Google Drive")
print("  2. Place in backend folder")
print("  3. Restart backend")
print("  4. Upload images OR videos!")
print("\n🎯 Detects ALL violations now!")
print("="*60)